# Week 7 / Lab 07 — Irregular Time Series and Quasar Reverberation Mapping
### Research-based learning · Graded Lab 4 (5% of course grade)
**MSc Astrostatistics for Observational Cosmology**

**Format:** integrated theory and 4-hour practical; complete the graded extension independently.
**Data:** first-year SDSS-RM light-curve products (Grier et al. 2017), VizieR J/ApJ/851/21.

> **Research question:** Can we infer a physically interpretable and robust emission-line
> delay from irregular, noisy AGN measurements, and decide whether it is suitable
> for subsequent quasar cosmology?

This notebook replaces the conventional Week 7 lecture. Work through the short
mathematical sections immediately before applying them. The original W07 lecture
remains a reference. No simulated astronomical catalogue replaces the published data.
The goal is not to force agreement with published lags or to obtain three detections.
A defensible non-detection earns full credit.

## Practical sequence and deliverables

| Contact time | Activity | Evidence |
|---|---|---|
| 0–25 min | Physical model, estimand, sampling | Five-line scientific statement |
| 25–55 min | Retrieve and audit real data | Three light-curve pairs and cadence table |
| 55–100 min | ICCF and overlap diagnostics | Peak, centroid, correlation curves |
| 100–145 min | FR/RSS | Lag distributions and uncertainty table |
| 145–155 min | Break | |
| 155–195 min | Robustness and failure analysis | Alternative windows and epoch removal |
| 195–220 min | Literature comparison | Observer/rest-frame comparison |
| 220–240 min | Cosmology connection and checkpoint | Decisions and limitations |

Read the required excerpts listed at the end. During the practical, complete A–F
and begin the core cosmology and literature checkpoints (H–H3 and I).
For submission, complete E1–E5, including a 300–500-word conclusion and the
conceptual SARM question in E4. Finish any incomplete core practical work independently.
Section G is background reading; the optional GP extension and the advanced
H4.1–H4.8 derivations and exercises are not assessed. Read the opening H4
paragraphs for the compulsory conceptual question; no SARM fit is required. Work with the same three objects, settings and
seed so that results are comparable. The supplied functions are infrastructure;
marks emphasize your independent analysis, diagnostics and reasoning.

# A. The physical and statistical problem
An accreting supermassive black hole produces a variable continuum. Ionizing
radiation drives a delayed broad emission-line response. A linear response model is

$$
L(t)=L_0+\int_0^\infty \Psi(u)C(t-u)\,du.
$$

$C$ is the driving continuum, $\Psi$ the transfer function and $L_0$ an offset.
If the transfer function is narrow, $L(t)\approx L_0+A C(t-\tau)$.
The observed optical continuum is a proxy for the unobserved ionizing continuum;
real BLRs have extended responses. A cross-correlation centroid is an estimator
of a characteristic delay, not an exact measurement of a unique geometric radius.

Observations are $c_i=C(t_i)+\epsilon_i$ and $\ell_j=L(s_j)+\eta_j$.
The times $t_i$ and $s_j$ need not coincide. Errors are heteroscedastic, and
calibration errors may be shared across epochs. The teaching resampling procedure
uses the published per-epoch uncertainties and assumes independent measurement errors;
it does not model all calibration covariance or an explicit transfer function.

**Sign convention:** $\tau>0$ means the line follows the continuum.
All lag searches use observer-frame days until explicitly converted.

**Stop and think:** Why is the lag not necessarily the separation between two
visually obvious maxima? Why is the optical continuum only a proxy?

## A1. Five-line scientific statement (before calculating lags)
Write 5–10 sentences in your own words:

1. **Estimand:** which time delay and which frame? What is its physical interpretation?
2. **Observed data:** what are the continuum and line quantities, epochs and uncertainties?
3. **Generative assumptions:** what response and noise assumptions are implicit?
4. **Uncertainty / selection:** what flags, band/telescope rule and limitations apply?
5. **Validation / falsification:** what observations would make you withhold a lag?

**Your response:** [Replace with your answer.]

# B. Data provenance and reproducible retrieval
Use the [CDS catalogue ReadMe](https://cdsarc.cds.unistra.fr/viz-bin/ReadMe/J/ApJ/851/21)
and [Grier et al. 2017](https://doi.org/10.3847/1538-4357/aa98dc).
Table 1 describes the 222 quasars; Table 2 contains 152,258 observations.
The data are extracted flux products, not raw spectra. Table 2 provides:

| Column | Meaning |
|---|---|
| RMID | Object identifier |
| MJD | MJD minus 50000, in days |
| Band | g, i, Halpha, Hbeta |
| Tel | C = CFHT, B = Bok, S = SDSS |
| Flux, e_Flux | Continuum flux density and error, in $10^{-17}$ erg s$^{-1}$ cm$^{-2}$ Å$^{-1}$ |
| S, e_S | Integrated line flux and error, in $10^{-17}$ erg s$^{-1}$ cm$^{-2}$ |
| Flag | 1 = rejected in the published analysis |
| z (Table 1) | Spectroscopic redshift |

Do not combine the two flux columns. Do not treat negative measured flux as
missing; retain finite values and strictly positive errors. Use Flag = 0.
The time offset is shared across bands and does not change delays.

**Three fixed objects:** RMID 101, 16, 33. These span different published Hβ
quality ratings (see Table 3 of Grier et al.). The exact performance under our
simplified single-band ICCF must be assessed from the data. Do not read the
published lag values until the literature-comparison section. The labels are
teaching design choices, not expected outcomes.

**Core continuum rule:** for each object, choose the g-band telescope with the
most distinct valid epochs (alphabetical tie break). This avoids pooling
uncalibrated telescope series. Hβ uses the released SDSS line measurements.
The published analysis used intercalibrated g+i continuum products; our
simplification is not an exact replication. Record which telescope is selected.

**Offline / Colab preparation.** Keep the notebook and `data_cache/sdss_rm_2017/`
in the same working directory. The first successful retrieval writes
`assigned_meta.csv` and `assigned_lc.csv`; retain these two files for offline reruns.
If your instructor provides these files, upload or extract them into that exact
folder before running the loader. In Colab, local runtime files disappear when
the runtime resets, so download a copy of the cache or retain it in Drive.
If using Drive, change `CACHE` in the setup cell to the mounted folder.
For an initial online run, install astroquery if necessary using `%pip install -q astroquery`
in a separate code cell. Never substitute synthetic observations for the catalogue.
The synthetic checks in D0 are algorithm tests only.

In [ ]:
# If astroquery is missing in Colab, run: %pip install -q astroquery
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy
from scipy.linalg import cho_factor, cho_solve
from IPython.display import display

IDS = (101, 16, 33)
SEED = 2026
MIN_PAIRS = 10
LAGS = np.arange(-100., 101., 1.)
ALT_LAGS = np.arange(-70., 71., 1.)
CENTROID_FRACTION = 0.8
MIN_R = 0.5   # teaching screen, NOT a significance threshold
B = 500       # final run; 100 for a quick trial, then restore 500
CACHE = Path('data_cache/sdss_rm_2017')
CACHE.mkdir(parents=True, exist_ok=True)
print(sys.version.split()[0], np.__version__, pd.__version__, scipy.__version__)

In [ ]:
# Supplied loader: use local CDS fixed-width files, or query VizieR by RMID.
# Manual fallback: download table1.dat and table2.dat (or .gz) from
# https://cdsarc.cds.unistra.fr/ftp/J/ApJ/851/21/ into CACHE.
# Fixed-width positions come directly from the CDS ReadMe.
T1_COLS = ['RMID', 'SDSS', 'RAdeg', 'DEdeg', 'z', 'imag', 'SNR',
           'logAGN', 'logHost', 'logM']
T1_SPEC = [(0,3),(4,23),(24,32),(33,40),(41,46),(47,53),
           (54,58),(59,63),(64,68),(69,72)]
T2_COLS = ['RMID','MJD','Band','Tel','Flux','e_Flux','S','e_S','Flag']
T2_SPEC = [(0,3),(4,13),(14,20),(21,22),(23,30),(31,37),
           (38,45),(46,52),(53,54)]

def local_table(stem, cols, spec):
    for suffix in ('.dat', '.dat.gz'):
        path = CACHE / (stem + suffix)
        if path.is_file():
            print('Reading', path)
            return pd.read_fwf(path, colspecs=spec, names=cols)
    return None

def query_one(table, cols, rmid):
    from astroquery.vizier import Vizier
    v = Vizier(columns=cols, row_limit=-1, timeout=120)
    result = v.query_constraints(catalog='J/ApJ/851/21/' + table,
                                 RMID=str(rmid))
    if len(result) != 1:
        raise RuntimeError(f'Unexpected VizieR result for {table}, RMID {rmid}')
    return result[0].to_pandas()

def fetch_tables():
    meta = local_table('table1', T1_COLS, T1_SPEC)
    obs = local_table('table2', T2_COLS, T2_SPEC)
    # Cache only the assigned real observations for offline re-runs.
    paths = [CACHE / 'assigned_meta.csv', CACHE / 'assigned_lc.csv']
    if meta is None:
        if paths[0].exists():
            print('Reading cached metadata:', paths[0])
            meta = pd.read_csv(paths[0])
        else:
            meta = pd.concat([query_one('table1', ['RMID','z'], i)
                              for i in IDS], ignore_index=True)
    if obs is None:
        if paths[1].exists():
            print('Reading cached observations:', paths[1])
            obs = pd.read_csv(paths[1])
        else:
            obs = pd.concat([query_one('table2', T2_COLS, i)
                             for i in IDS], ignore_index=True)
    for df in (meta, obs):
        df['RMID'] = pd.to_numeric(df['RMID'], errors='raise').astype(int)
    meta = meta.loc[meta.RMID.isin(IDS)].copy()
    obs = obs.loc[obs.RMID.isin(IDS)].copy()
    assert set(meta.RMID) == set(IDS) and set(obs.RMID) == set(IDS)
    assert not meta.RMID.duplicated().any()
    assert set(T2_COLS).issubset(obs.columns)
    for col in ['Band', 'Tel']:
        obs[col] = obs[col].map(lambda x: x.decode() if isinstance(x, bytes) else str(x)).str.strip()
    meta.to_csv(paths[0], index=False)
    obs.to_csv(paths[1], index=False)
    return meta, obs

try:
    meta, obs = fetch_tables()
except Exception as exc:
    raise RuntimeError('Real CDS data unavailable. Install astroquery, retry '
                       'VizieR, or download the original fixed-width tables '
                       f'into {CACHE.resolve()}. Do not invent data.') from exc

for col in ['MJD','Flux','e_Flux','S','e_S','Flag']:
    obs[col] = pd.to_numeric(obs[col], errors='coerce')
for col in ['Band','Tel']:
    obs[col] = obs[col].map(lambda x: x.decode() if isinstance(x, bytes) else str(x))
    obs[col] = obs[col].str.strip()
meta['z'] = pd.to_numeric(meta['z'], errors='raise')
assert np.isfinite(meta.z).all() and (meta.z > 0).all()
display(meta[['RMID','z']])
display(obs.groupby(['RMID','Band','Tel']).size().rename('raw_rows').reset_index())

## B1. Cleaning and duplicate epochs
For observations with exactly identical timestamps within a selected series,
combine by inverse-variance weighting, assuming independent measurement errors:

$$
\bar f=\frac{\sum_i f_i/\sigma_i^2}{\sum_i1/\sigma_i^2},\qquad
\sigma_{\bar f}=(\sum_i1/\sigma_i^2)^{-1/2}.
$$

This is a declared approximation; correlated calibration can make the errors
too small. Near-identical times are not binned. Use one common time origin per
object, never subtract a separate minimum from the two light curves (doing so
would shift their relative timing). Record all cleaning counts.

In [ ]:
def clean_band(raw, band, flux, err):
    a = raw.loc[(raw.Band == band) & (raw.Flag == 0)].copy()
    ok = (np.isfinite(a.MJD) & np.isfinite(a[flux]) &
          np.isfinite(a[err]) & (a[err] > 0))
    return a.loc[ok].copy()

def combine_epochs(a, flux, err, t0):
    b = pd.DataFrame({'t': a.MJD.to_numpy(float) - t0,
                      'f': a[flux].to_numpy(float),
                      'e': a[err].to_numpy(float)})
    b['w'] = 1 / b.e**2
    b['wf'] = b.w * b.f
    s = b.groupby('t', sort=True)[['w','wf']].sum()
    return np.column_stack([s.index.to_numpy(float), s.wf/s.w,
                            1/np.sqrt(s.w)])

def make_pair(rmid, band='g'):
    raw = obs.loc[obs.RMID == rmid]
    cont = clean_band(raw, band, 'Flux', 'e_Flux')
    line = clean_band(raw, 'Hbeta', 'S', 'e_S')
    line = line.loc[line.Tel == 'S']
    counts = cont.groupby('Tel').MJD.nunique()
    if counts.empty:
        raise ValueError(f'No valid {band} continuum for RMID {rmid}')
    tel = sorted(counts.index[counts == counts.max()])[0]
    cont = cont.loc[cont.Tel == tel]
    if min(cont.MJD.nunique(), line.MJD.nunique()) < MIN_PAIRS:
        raise ValueError(f'Insufficient valid epochs for RMID {rmid}')
    t0 = min(cont.MJD.min(), line.MJD.min())
    return {'c': combine_epochs(cont, 'Flux', 'e_Flux', t0),
            'l': combine_epochs(line, 'S', 'e_S', t0),
            'z': float(meta.loc[meta.RMID == rmid,'z'].iloc[0]),
            'tel': tel, 'band': band, 't0': t0,
            'raw_c': len(raw.loc[(raw.Band == band)&(raw.Tel == tel)]),
            'raw_l': len(raw.loc[(raw.Band == 'Hbeta')&(raw.Tel == 'S')]),
            'valid_c': len(cont), 'valid_l': len(line)}

pairs = {i: make_pair(i) for i in IDS}

# C. Irregular sampling and variability
For ordered distinct times, the gaps are $\Delta t_j=t_{j+1}-t_j$. A single
median cadence is not sufficient: long gaps may lie between many short ones.
The baseline is $T=t_{\max}-t_{\min}$. Compare lag scales with both cadence
and baseline; a one-day search step does not imply one-day time resolution.

A descriptive excess-variance measure is

$$
F_{\rm var}=\frac{\sqrt{s_f^2-\langle\sigma_i^2\rangle}}{\bar f},
$$

when the excess variance and mean are positive. Otherwise report undefined.
This simple estimate does not account for red noise, covariance or sampling
bias. Do not infer lack of intrinsic variability merely from undefined Fvar.

In [ ]:
def audit(a):
    t, f, e = a.T
    gaps = np.diff(t)
    excess = np.var(f, ddof=1) - np.mean(e**2)
    return {'N': len(a), 'baseline_d': np.ptp(t),
            'median_gap_d': np.median(gaps), 'max_gap_d': gaps.max(),
            'median_error': np.median(e),
            'Fvar': np.sqrt(excess)/np.mean(f)
            if excess > 0 and np.mean(f) > 0 else np.nan}

rows = []
for i,p in pairs.items():
    for k, label in [('c', p['band']+' / '+p['tel']), ('l','Hbeta / S')]:
        rows.append({'RMID':i, 'series':label, 'raw_rows':p['raw_'+k],
                     'valid_rows':p['valid_'+k], **audit(p[k])})
audit_table = pd.DataFrame(rows)
display(audit_table.round(4))

In [ ]:
def plot_pair(i,p):
    fig, ax = plt.subplots(2,1,figsize=(9,5),sharex=True,
                           constrained_layout=True)
    for a,k,label in zip(ax,['c','l'],['Continuum flux density','Hbeta line flux']):
        t,f,e=p[k].T
        a.errorbar(t,f,yerr=e,fmt='.',capsize=2,alpha=.8)
        a.set_ylabel(label + '\n[catalogue units]')
    ax[0].set_title(f'RMID {i}, z={p["z"]:.3f}, {p["band"]}/{p["tel"]}')
    ax[1].set_xlabel('Observer-frame days from common origin')
    plt.show()
for i,p in pairs.items():
    plot_pair(i,p)

### Exercise C1 — inspection before inference
Write a 100–150-word prediction of which object might be hardest to analyse,
using actual cadence, maximum gaps, baseline, error bars and variability.
Do not look at the published lag values. Plot the gap distributions for the
continuum and Hβ of each object (shared units, appropriate bins). Explain
why apparent smoothness or a large number of photometric points is not
sufficient: sparse line measurements may limit the result.

In [ ]:
# Student workspace: plot gap distributions.
# Add your prediction in a markdown cell below.

# D. Interpolation cross-correlation (ICCF)
Define the Pearson correlation at trial delay $\tau$ by aligning the continuum
with the later line response:

$$
r(\tau)=\operatorname{Corr}[C(t),L(t+\tau)].
$$

In one direction interpolate the continuum at $s_j-\tau$ and correlate with
measured line flux at $s_j$. In the other direction interpolate the line at
$t_i+\tau$ and correlate with measured continuum at $t_i$. The average of the
two local Pearson coefficients gives our two-way ICCF. Local means and
standard deviations are recomputed at every lag, only over valid pairs.
No extrapolation; at least 10 pairs per direction. Interpolated values are
not independent observations. Long internal gaps are not excluded in the
baseline but are diagnosed via sampling and optional gap-limited sensitivity.

$$
r_k=\frac{\sum_j(x_j-\bar x_k)(y_j-\bar y_k)}
{\sqrt{\sum_j(x_j-\bar x_k)^2\sum_j(y_j-\bar y_k)^2}}.
$$

The symmetric search interval is [-100,100] observer-frame days, step 1 day.
It is fixed before viewing results. Negative lags are retained as a diagnostic.
A positive peak is not automatically evidence of reverberation. Inspect overlap,
edge effects, slow trends, red noise and competing peaks.

In [ ]:
# Vectorized local correlations: one row per trial lag.
def directional(t_source, f_source, query, measured, max_gap=None):
    y = np.interp(query.ravel(), t_source, f_source,
                  left=np.nan, right=np.nan).reshape(query.shape)
    if max_gap is not None:
        j = np.searchsorted(t_source, query, side='left')
        lo = np.clip(j-1, 0, len(t_source)-1)
        hi = np.clip(j, 0, len(t_source)-1)
        exact = t_source[hi] == query
        y[(t_source[hi]-t_source[lo] > max_gap) & ~exact] = np.nan
    valid = np.isfinite(y)
    count = valid.sum(axis=1)
    denom = np.maximum(count,1)[:,None]
    x = np.broadcast_to(measured, y.shape)
    xmean = np.where(valid,x,0).sum(axis=1)[:,None]/denom
    ymean = np.where(valid,y,0).sum(axis=1)[:,None]/denom
    dx = np.where(valid,x-xmean,0)
    dy = np.where(valid,y-ymean,0)
    norm = np.sqrt((dx*dx).sum(axis=1)*(dy*dy).sum(axis=1))
    r = np.full(len(count),np.nan)
    ok = (count >= MIN_PAIRS) & (norm > 0)
    r[ok] = np.clip((dx*dy).sum(axis=1)[ok]/norm[ok],-1,1)
    return r,count

def iccf(c,l,lags=LAGS,max_gap=None):
    tc,fc,_=c.T; tl,fl,_=l.T
    r1,n1=directional(tc,fc,tl[None,:]-lags[:,None],fl,max_gap)
    r2,n2=directional(tl,fl,tc[None,:]+lags[:,None],fc,max_gap)
    return (r1+r2)/2, r1,r2,n1,n2

def lag_summary(lags,r):
    if not np.isfinite(r).any():
        return dict(peak=np.nan,cent=np.nan,rmax=np.nan,edge=True)
    k=int(np.nanargmax(r))
    peak=float(lags[k]); rmax=float(r[k])
    a=b=k
    if rmax > 0:
        threshold=CENTROID_FRACTION*rmax
        while a>0 and np.isfinite(r[a-1]) and r[a-1]>=threshold:
            a-=1
        while b<len(r)-1 and np.isfinite(r[b+1]) and r[b+1]>=threshold:
            b+=1
    edge=(a==0 or b==len(r)-1 or
          (a>0 and not np.isfinite(r[a-1])) or
          (b<len(r)-1 and not np.isfinite(r[b+1])))
    cent=float(np.average(lags[a:b+1],weights=r[a:b+1])) if rmax>0 else np.nan
    return dict(peak=peak,cent=cent,rmax=rmax,edge=bool(edge))

## D0. Validate the estimator before interpreting real observations
Run the supplied deterministic unit tests. The artificial signals below are **only numerical fixtures**, not simulated AGN observations and not part of the graded astronomical sample. They check the lag sign, reversal symmetry, search-boundary flag, and minimum-overlap rule. A passing test verifies these implementation properties, not statistical significance or recovery on noisy, irregular SDSS-RM data. Keep the test output in your submission. If a test fails, diagnose it before proceeding.

In [ ]:
def validate_iccf():
    rng = np.random.default_rng(314159)
    t = np.arange(160., dtype=float)
    # Distinctive reproducible signal; exact shifted copy on a translated grid.
    f = rng.normal(size=t.size)
    err = np.full_like(t, 0.1)
    delay = 12.0
    c = np.column_stack((t, f, err))
    l = np.column_stack((t + delay, f, err))
    grid = np.arange(-30., 31.)
    r = iccf(c, l, grid)[0]
    forward = lag_summary(grid, r)
    reverse_r = iccf(l, c, grid)[0]
    reverse = lag_summary(grid, reverse_r)
    assert forward['peak'] == delay and abs(forward['cent']-delay) <= 1
    assert not forward['edge'] and np.isclose(forward['rmax'], 1)
    assert reverse['peak'] == -delay and abs(reverse['cent']+delay) <= 1
    assert np.allclose(r, reverse_r[::-1], equal_nan=True)
    clipped_grid = np.arange(-10., delay+1)
    clipped = lag_summary(clipped_grid, iccf(c, l, clipped_grid)[0])
    assert clipped['peak'] == delay and clipped['edge']
    short_c, short_l = c[:MIN_PAIRS-1], l[:MIN_PAIRS-1]
    short_r, _, _, n1, n2 = iccf(short_c, short_l, grid)
    assert np.all(n1 < MIN_PAIRS) and np.all(n2 < MIN_PAIRS)
    assert np.isnan(short_r).all()
    assert np.isnan(lag_summary(grid, short_r)['cent'])
    print('PASS: positive delay, reversed sign, boundary flag, insufficient overlap.')

validate_iccf()

## D1. Peak and centroid
The peak lag is $\tau_{\rm peak}=\arg\max_\tau r(\tau)$.
The centroid is a correlation-weighted mean of the **contiguous** region around
the global maximum with $r\geq0.8r_{\max}$:

$$
\tau_{\rm cent}=\frac{\sum_{k\in A}\tau_k r_k}{\sum_{k\in A}r_k}.
$$

Disconnected peaks are not averaged together. If the centroid region touches
the search boundary or an undefined correlation, mark it as edge-limited.
Ties are resolved at the first grid maximum (report obvious ties). We record
all raw estimates but use a predeclared screen $r_{\max}\geq0.5$ and no edge
clipping when summarizing FR/RSS. This is a teaching quality screen, not a
p-value or the published selection criterion. A centroid can be negative.

In [ ]:
def plot_iccf(i,p,lags=LAGS):
    r,r1,r2,n1,n2=iccf(p['c'],p['l'],lags)
    s=lag_summary(lags,r)
    fig,ax=plt.subplots(2,1,sharex=True,figsize=(8,5),
                        constrained_layout=True)
    ax[0].plot(lags,r,label='Two-way ICCF',lw=2)
    ax[0].plot(lags,r1,ls=':',label='Continuum interpolated')
    ax[0].plot(lags,r2,ls='--',label='Line interpolated')
    if np.isfinite(s['cent']):
        ax[0].axvline(s['cent'],ls='--',color='k',label='Centroid')
        ax[0].axhline(.8*s['rmax'],ls=':',color='.5')
    ax[0].axvline(0,color='.5',lw=.7)
    ax[0].set(ylabel='Local Pearson r',title=f'RMID {i}')
    ax[0].legend(fontsize=8)
    ax[1].plot(lags,n1,label='Measured line pairs')
    ax[1].plot(lags,n2,label='Measured continuum pairs')
    ax[1].axhline(MIN_PAIRS,ls=':',color='.5')
    ax[1].set(xlabel='Observer-frame lag [days]',ylabel='Overlap pairs')
    ax[1].legend(fontsize=8)
    plt.show()
    return s

base = {i:plot_iccf(i,p) for i,p in pairs.items()}
display(pd.DataFrame(base).T)

### Exercise D2 — interpret the correlation curves
For each object, describe whether the two directions agree, whether the peak
is broad or has competitors, and whether overlap decreases at the peak.
Compare peak and centroid. Identify any negative or edge-limited result.
Do not decide on a lag solely because the ICCF maximum exceeds 0.5.
Explain why a smaller lag grid step cannot recover information absent from the data.

# E. Lag uncertainty: flux randomization and random subset selection
FR/RSS (Peterson et al. 1998; see also Welsh 1999) perturbs the observed light
curves to examine estimator sensitivity. In every realization:

1. independently draw N epoch indices with replacement for each series;
2. retain distinct epochs, with multiplicity $m_i$;
3. randomize each retained flux once with Gaussian error $\sigma_i/\sqrt{m_i}$;
4. recompute the same two-way ICCF and the contiguous centroid;
5. retain the centroid only if the stated screen is satisfied.

This is the commonly used multiplicity-adjusted FR/RSS convention. RSS does
not reconstruct missing seasons. Measurement errors are assumed independent
and the temporal dependence of the underlying AGN is not a bootstrap model.
FR/RSS estimates a **conditional resampling distribution**, not a Bayesian
posterior or a false-alarm probability. Report failed and edge-limited trials,
negative-lag fractions among accepted draws, and broad or multimodal histograms.
A narrow conditional distribution after discarding many draws is not sufficient.

In [ ]:
def fr_rss(a,rng):
    ix=rng.integers(0,len(a),size=len(a))
    unique,mult=np.unique(ix,return_counts=True)
    b=a[unique].copy()
    b[:,2]/=np.sqrt(mult)
    b[:,1]+=rng.normal(0,b[:,2])
    return b

def resample_lags(p,lags=LAGS,B=B,seed=SEED):
    rng=np.random.default_rng(seed)
    records=[]
    for j in range(B):
        cc=fr_rss(p['c'],rng); ll=fr_rss(p['l'],rng)
        r,*_=iccf(cc,ll,lags)
        s=lag_summary(lags,r)
        s['accepted']=bool(np.isfinite(s['cent']) and
                           s['rmax']>=MIN_R and not s['edge'])
        records.append(s)
    return pd.DataFrame(records)

def summarize_draws(draws):
    accepted=draws.loc[draws.accepted,'cent'].to_numpy(float)
    q=np.quantile(accepted,[.16,.5,.84]) if len(accepted) else [np.nan]*3
    return dict(B=len(draws), accepted=len(accepted),
                acceptance=len(accepted)/len(draws),
                edge_fraction=draws.edge.mean(),
                nonpositive_fraction=np.mean(accepted<=0) if len(accepted) else np.nan,
                q16=q[0],q50=q[1],q84=q[2])

draws={i:resample_lags(p,seed=SEED+i) for i,p in pairs.items()}
uncertainty=pd.DataFrame({i:summarize_draws(d) for i,d in draws.items()}).T
uncertainty.index.name='RMID'
display(uncertainty.round(4))

In [ ]:
fig,axes=plt.subplots(len(IDS),1,figsize=(8,7),sharex=True,
                      constrained_layout=True)
for ax,i in zip(axes,IDS):
    d=draws[i]; good=d.loc[d.accepted,'cent']
    ax.hist(good,bins=np.arange(-100,105,5),histtype='stepfilled',alpha=.55)
    ax.axvline(0,color='.4',ls=':')
    if len(good):
        q=np.quantile(good,[.16,.5,.84])
        ax.axvline(q[1],color='k',ls='--')
        ax.axvspan(q[0],q[2],alpha=.15)
    ax.set(ylabel='Accepted draws',title=f'RMID {i}: {len(good)}/{len(d)} accepted')
axes[-1].set_xlabel('Observer-frame centroid [days]')
plt.show()

### Exercise E0 — uncertainty and numerical repeatability
For all three objects, describe the centroid distributions and report their
accepted/total trials, failed fraction and nonpositive fraction. Explain
whether a central interval is a useful summary or conceals multiple modes.
Repeat FR/RSS with a second seed for one object; compare quantiles and
acceptance. If estimates fluctuate substantially, increase B and report the
final B used. Do not interpret a resampling interval as a detection probability.

In [ ]:
# Student workspace: second seed and numerical comparison.

# F. Robustness and failure analysis (continuation of Week 6)
We change one analysis choice at a time. Prespecify the following:

- Alternative symmetric search window [-70,70] days, same step and estimator.
- Remove 20% of the distinct epochs in each series, independently and without
  replacement, using fixed seed 2026 + RMID. Repeat across 20 deletion seeds.
- Keep the same band, telescope, flag rule and time origin in both experiments.

The smaller window may exclude real lags; it is a sensitivity test rather
than a preferred result. Epoch deletion assesses robustness to available
measurements, not a calibrated selection correction. If an experiment produces
no valid estimate, record that explicitly. Do not rerun until the answer agrees.

In [ ]:
def delete_epochs(a,rng,fraction=.20):
    nremove=int(np.floor(fraction*len(a)))
    keep=np.sort(rng.choice(len(a),len(a)-nremove,replace=False))
    return a[keep]

def deletion_trials(p,rmid,ntrial=20):
    records=[]
    for j in range(ntrial):
        rng=np.random.default_rng(SEED+rmid+1000*j)
        cc=delete_epochs(p['c'],rng); ll=delete_epochs(p['l'],rng)
        r,*_=iccf(cc,ll,LAGS)
        records.append(lag_summary(LAGS,r))
    d=pd.DataFrame(records)
    d['accepted']=(d.rmax>=MIN_R)&(~d.edge)&np.isfinite(d.cent)
    return d

alt={i:lag_summary(ALT_LAGS,iccf(p['c'],p['l'],ALT_LAGS)[0])
     for i,p in pairs.items()}
deletions={i:deletion_trials(p,i) for i,p in pairs.items()}
robust=[]
for i in IDS:
    d=deletions[i]; a=d.loc[d.accepted,'cent']
    robust.append(dict(RMID=i,base_cent=base[i]['cent'],
                       base_edge=base[i]['edge'],alt_cent=alt[i]['cent'],
                       alt_rmax=alt[i]['rmax'],alt_edge=alt[i]['edge'],
                       alt_shift=alt[i]['cent']-base[i]['cent'],
                       deletion_valid=int(d.accepted.sum()),
                       deletion_median=a.median(),
                       deletion_min=a.min(),deletion_max=a.max()))
robust_table=pd.DataFrame(robust)
display(robust_table.round(3))

### Exercise F1 — student-led diagnostic analysis
Plot the baseline and alternative ICCF for every object with common lag axes.
Plot the 20 deletion results, including failures (annotate counts rather than
plotting fabricated values). Explain whether shifts are small relative to the
resampling interval and what the shift means. If the centroid becomes
edge-limited, or a competing peak becomes dominant, say so.

For your most ambiguous object, add one justified diagnostic: compare i-band
continuum using the same telescope-selection rule, inspect the impact of long
interpolation gaps, or compare the two ICCF directions. State the choice before
calculating it. Compare results without silently replacing the prescribed baseline.

In [ ]:
# Student workspace: robustness comparison and additional failure diagnostic.

# G. Red noise and Gaussian processes: why correlation is not significance
AGN light curves exhibit temporal dependence. A useful first model is an
Ornstein–Uhlenbeck (damped random walk) process:

$$
k(t,t')=\sigma_{\rm DRW}^2\exp[-|t-t'|/T_{\rm DRW}].
$$

The covariance of the noisy measurements is $K_{ij}=k(t_i,t_j)+\delta_{ij}\sigma_i^2$.
Slow independent stochastic trends can create strong correlations over a
finite baseline. The ordinary Pearson p-value assumes independent pairs and
is not an appropriate false-alarm test here. FR/RSS keeps the observed
variability pattern and is not a null test. A calibrated significance study
would need independent red-noise realizations with appropriate covariance,
sampling, noise and the same search procedure (including the look-elsewhere
 effect). That null analysis is outside the compulsory real-data lab.

A GP describes the latent continuum with a predictive distribution rather
than a deterministic interpolation. For fixed hyperparameters, mean vector
$m$, observed values $y$ and prediction times $t_*$,

$$
\mathbb E[f_*|y]=m_*+K_{*t}(K_{tt}+\Sigma)^{-1}(y-m_t),
$$
$$
\operatorname{Cov}(f_*|y)=K_{**}-K_{*t}(K_{tt}+\Sigma)^{-1}K_{t*}.
$$

These formulas show why uncertainty increases away from constraining epochs.
A GP continuum prediction is not itself a full reverberation model. JAVELIN
(Zu et al. 2011) jointly models a stochastic continuum and a delayed, smoothed
response; its lag distribution has different assumptions from ICCF FR/RSS.

**Optional:** work through the [Crete Gaussian Processes notebook](https://github.com/lionandjelka/2025_summer_school/blob/main/08_Gaussian_Processes/Gaussian_Processes.ipynb).
Fit an OU GP to a real continuum, compare predictive uncertainty in a gap,
and discuss kernel dependence. Do not use simulated data in place of SDSS-RM.

# H. Rest-frame lag, physical radius and quasar cosmology
Cosmological time dilation gives

$$
\tau_{\rm rest}=\frac{\tau_{\rm obs}}{1+z},\qquad
R_{\rm BLR}\simeq c\tau_{\rm rest}.
$$

A rest-frame lag of one day corresponds to one light-day. Divide **all**
quantiles by $1+z$ (assuming negligible redshift error), not just the median.
A negative or ambiguous delay should not be converted into a physical radius
or logged. For teaching transparency, retain signed lag summaries in the
comparison table and assign physical radii only to justified usable lags.

In [ ]:
rest_rows=[]
for i in IDS:
    z=pairs[i]['z']; q=uncertainty.loc[i,['q16','q50','q84']].to_numpy(float)
    rest=q/(1+z)
    rest_rows.append(dict(RMID=i,z=z,obs16=q[0],obs50=q[1],obs84=q[2],
                          rest16=rest[0],rest50=rest[1],rest84=rest[2]))
rest_table=pd.DataFrame(rest_rows)
display(rest_table.round(4))

## H1. From reverberation mapping to a standardizable distance indicator
For the Hβ line the empirical radius–luminosity relation is often written
(Bentz et al. 2013):

$$
\log_{10}(\tau_{\rm rest}/\mathrm{day})=K+\alpha\log_{10}(L_{5100}/L_{\rm ref})+\epsilon.
$$

Here $L_{5100}=\lambda L_\lambda(5100\,\mathrm{\AA})$ is the host-subtracted
monochromatic luminosity in erg/s; the intrinsic scatter $\epsilon$ describes
astrophysical departures. The slope is near one-half in the local Hβ calibration,
not a universal fixed law for every line or accretion state. Watson et al. (2011)
and Czerny et al. (2013) explore how this relation can provide luminosity distances.

For a calibrated intercept and slope,

$$
L_{5100}=L_{\rm ref}\,10^{[\log_{10}(\tau_{\rm rest}/\mathrm{day})-K]/\alpha},
$$
$$
D_L=\sqrt{\frac{L_{5100}}{4\pi F_{5100}}},\qquad
\mu=5\log_{10}(D_L/\mathrm{Mpc})+25.
$$

$F_{5100}$ is a consistently defined, extinction- and host-corrected monochromatic
flux corresponding to rest 5100 Å; it is not the integrated Hβ line flux or
an arbitrary g-band measurement. With matched observed wavelength
$\lambda_{\rm obs}=(1+z)5100$ Å, $F_{5100}=\lambda_{\rm obs}f_\lambda(\lambda_{\rm obs})$.
Do not substitute the published `logAGN` as an independent luminosity: it already
depends on a distance scale. Lab 07 lacks the calibrated flux and population
model required for an independent cosmological distance measurement.

## H2. Why lag reliability matters quantitatively
At fixed calibration and flux, $D_L\propto\tau_{\rm rest}^{1/(2\alpha)}$.
For small independent errors, the lag-only contribution is

$$
\sigma_{\ln D_L,\tau}\approx\frac{\sigma_\tau}{2|\alpha|\tau},\qquad
\sigma_{\mu,\tau}\approx\frac{5}{\ln 10}\frac{\sigma_\tau}{2|\alpha|\tau}.
$$

For asymmetric positive quantiles, use the exact transformation instead:

$$
\Delta\mu_{16,84}=\frac{5}{2\alpha}\log_{10}(\tau_{16,84}/\tau_{50}),\quad \alpha>0.
$$

This isolates the effect of lag uncertainty without pretending that a flux,
absolute luminosity or distance was measured. It excludes calibration,
flux, extinction, intrinsic R–L scatter, selection and their covariance.
For a nonpositive lower quantile or multimodal ambiguous lag, do not force a
log-distance summary. Such an object is a poor candidate for this calculation.

**Student exercise:** for each lag you regard as usable, adopt the illustrative
$\alpha=0.5$ (not a fitted calibration) and calculate the asymmetric lag-only
relative distance-modulus shifts. Explain why a symmetric lag error does not
necessarily give a symmetric magnitude error.

In [ ]:
# Student workspace: set your accepted RMIDs only AFTER diagnostics.
usable_ids = []  # replace with your justified choice(s); [] is allowed
alpha = 0.5
for i in usable_ids:
    row=rest_table.loc[rest_table.RMID==i].iloc[0]
    q=row[['rest16','rest50','rest84']].to_numpy(float)
    if not np.isfinite(q).all() or np.any(q<=0):
        print(i,'No valid logarithmic lag-only propagation')
        continue
    shifts=5/(2*alpha)*np.log10(q[[0,2]]/q[1])
    print(i,'radius [light-days]:',q[1],
          'lag-only delta distance modulus [mag]:',shifts)

## H3. Quasars in cosmology: two luminosity-based approaches
Reverberation distances use the time-delay/size and luminosity relation. A
separate method uses the non-linear UV–X-ray luminosity relation,

$$
\log L_X=\beta+\gamma\log L_{UV}.
$$

Using consistently defined rest-frame fluxes $L_j=4\pi D_L^2F_j$ gives

$$
\log F_X=\beta+(\gamma-1)\log(4\pi)+\gamma\log F_{UV}
+2(\gamma-1)\log D_L.
$$

The latter method does not need BLR time lags. Risaliti & Lusso (2019)
illustrate high-redshift applications; subsequent studies examine calibration,
redshift dependence and systematic differences between quasar distance methods.
These are research probes, not interchangeable or fully established standard candles. Section H4 introduces the distinct geometric SARM approach.

The R–L intercept and cosmological distances are degenerate unless calibrated.
A cosmological analysis should jointly fit relation parameters, intrinsic scatter
and cosmology, or use independent calibration with propagated uncertainty.
Selection by flux, lag detectability and accretion state may alter the observed
population. The same warning from Week 6 applies: selecting apparently reliable
lags is not a formal selection-function correction.

**Discussion:** Why do quasars extend the redshift range beyond many SNe Ia?
Why does that not automatically make them more precise distance indicators?
How could removing uncertain lags introduce selection bias in an R–L sample?

## H4. SARM: a geometric alternative to quasar standardization

**Core concept below; advanced independent-study extension H4.1–H4.8 (45–60 minutes, optional).** Read the abstract
of Wang et al. (2020) and Li et al. (2025); then work through the following
sections. Only the conceptual SARM question in E4 is compulsory; the detailed derivations, published-result plots and numerical exercise below are optional.
No interferometric observations are supplied for the three SDSS-RM targets.
This section uses published SARM results, not a new fit to those targets.

SpectroAstrometry and Reverberation Mapping (SARM) combines two complementary
measurements of the broad-line region (BLR):

- RM: light-travel delays constrain a physical length scale.
- SA: wavelength-dependent photocentres constrain angular structure.

For a consistently defined physical radius $R$ and angular radius $\theta$
(in radians), the small-angle relation is

$$D_A=\frac{R}{\theta},\qquad R\sim c\tau_{\rm rest}.$$

The observed-frame lag must be converted to the source frame before using it
as a light-travel scale. SARM provides an **angular-diameter distance**, not
initially a luminosity distance. Unlike the R–L method, it needs no empirical
luminosity calibration. Unlike the UV–X-ray method, it uses spatial and temporal
information together. Geometric does not mean free of astrophysical assumptions.

For intuition only, since one light-day is $c\times86400$ s and one microarcsecond
is $\pi/(180\times3600\times10^6)$ radians,

$$D_A[\mathrm{Mpc}]\simeq173.14\frac{R[\mathrm{light\! -\! days}]}{\theta[\mu\mathrm{as}]}.$$

This ratio is valid for matched physical and angular scales, not an arbitrary
ICCF centroid divided by an observed differential photocentre.

### H4.1. Why reverberation mapping is a geometric measurement

Let $\mathbf r$ denote a gas position relative to the continuum source and
$\hat{\mathbf n}$ the unit vector from the source toward the observer. For a
point-like central ionizing source and a distant observer, the extra path is
$r-\hat{\mathbf n}\cdot\mathbf r$. The rest-frame delay is therefore

$$u(\mathbf r)=\frac{r-\hat{\mathbf n}\cdot\mathbf r}{c}.$$

For gas at the near side, the delay may be small even when its radius is
large. Gas on the far side may have a delay approaching $2r/c$. The radius
and delay of an individual cloud are not identical quantities.

A velocity-resolved transfer function can be represented schematically as

$$\Psi(v,u)=\int W_{\rm RM}(\mathbf r,\mathbf w)\,
\delta(v-v_{\rm los})\,\delta[u-u(\mathbf r)]\,d^3r\,d^3w,$$

where $\mathbf w$ is the gas velocity, $v_{\rm los}$ uses a declared Doppler
sign convention and $W_{\rm RM}$ includes the variable line responsivity.
The observable variable line profile is

$$\Delta L(v,t)=\int_0^\infty\Psi(v,u)\Delta C(t-u)\,du.$$

All times in this expression are in the source frame (or consistently
transformed to observed time). Integrating over $v$ gives the ordinary
one-dimensional reverberation response. A velocity-delay map adds information
about geometry and kinematics; it is not obtained from integrated line light
curves alone. The ICCF centroid in the core lab is an estimator of a delay
and depends on the continuum variability, sampling and transfer function.
It is not a direct measurement of a unique BLR radius.

### H4.2. Spectroastrometry and interferometric differential phases

A spectral channel selects gas with a range of projected velocities. For an
emission-line channel centred on $v$, define the line-emission-weighted
physical photocentre projected onto the sky:

$$\mathbf x_{\ell}(v)=\frac{\int \mathbf r_\perp W_{\rm SA}
\delta(v-v_{\rm los})\,d^3r\,d^3w}{\int W_{\rm SA}
\delta(v-v_{\rm los})\,d^3r\,d^3w}.$$

Its angular position is $\boldsymbol\epsilon_\ell(v)=\mathbf x_\ell(v)/D_A$.
For a marginally resolved source, adopt the Fourier convention
$V(\mathbf u)=\int I(\boldsymbol\epsilon)e^{-2\pi i\mathbf u\cdot
\boldsymbol\epsilon}d^2\epsilon/\int I d^2\epsilon$ with
$\mathbf u=\mathbf B_\perp/\lambda$. Taylor expansion gives a first-order
visibility phase proportional to the centroid:

$$\phi\simeq-2\pi\,\mathbf u\cdot\boldsymbol\epsilon.$$

For a continuum-referenced channel, in the marginally resolved limit,

$$\Delta\phi(v)\simeq-\frac{2\pi}{\lambda}\mathbf B_\perp\cdot
\left[\frac{F_\ell(v)}{F_\ell(v)+F_c(v)}
(\boldsymbol\epsilon_\ell(v)-\boldsymbol\epsilon_c)\right].$$

$\mathbf B_\perp$ is the projected interferometric baseline, $\lambda$ is
observed wavelength, and $F_\ell,F_c$ are channel flux contributions. The
line-to-total flux ratio dilutes the phase; this expression assumes both
components are in the photocentre regime. If the continuum or line is
significantly resolved, use the full complex-visibility model. Phase signs
also depend on the baseline/Fourier convention.

A rotating BLR can display opposite red/blue photocentre displacements.
Several baseline orientations and spectral channels help constrain the
projected geometry. A measured phase alone is not a radius: projection,
continuum dilution and the spatial emission distribution must be modelled.
GRAVITY Collaboration (2018) demonstrated this on the Pa$\alpha$ line of 3C 273.

**Stop and think:** Why can a strong rotating velocity field produce nearly
zero differential phase on a particular projected baseline? Why is the
maximum photocentre offset not necessarily the mean angular BLR radius?

### H4.3. Joint Bayesian inference: two datasets, shared physics

A pedagogical SARM parameter vector is

$$\boldsymbol\vartheta=(D_A,M_\bullet,i,\mathrm{PA},
\boldsymbol\eta_{\rm geom},\boldsymbol\eta_{\rm kin},
\boldsymbol\eta_{\rm emiss},\boldsymbol\eta_{\rm resp},
\boldsymbol\eta_C,\boldsymbol\eta_{\rm cal}).$$

The symbols describe distance, black-hole mass, viewing inclination, sky
position angle, geometry, dynamics, emission and response weights, latent
continuum parameters and calibration/nuisance parameters. A dynamical model
predicts both the reverberation observations and the interferometric spectra
and phases. For example, circular Keplerian orbits would have
$v_{\rm K}(r)=\sqrt{GM_\bullet/r}$, but inflow, outflow, turbulence and a thick
BLR can modify the predictions.

If the RM and SA observational errors are conditionally independent given
all shared model and calibration parameters,

$$p(\boldsymbol\vartheta\mid\mathcal D_{\rm RM},\mathcal D_{\rm SA})
\propto\mathcal L_{\rm RM}(\boldsymbol\vartheta)
\mathcal L_{\rm SA}(\boldsymbol\vartheta)p(\boldsymbol\vartheta).$$

For example, each Gaussian likelihood includes a residual quadratic form
and its covariance normalization; any parameter-dependent covariance must
retain its log determinant. If calibration or continuum uncertainties couple
the datasets, include the shared latent variables or use a joint covariance.
The marginal distance posterior integrates over the other parameters,

$$p(D_A\mid\mathcal D)=\int p(D_A,\boldsymbol\eta\mid\mathcal D)\,d\boldsymbol\eta.$$

The joint posterior is important: SA constrains angular scale and projected
velocities, while RM introduces physical scale and velocity-delay structure.
Their complementary degeneracies can constrain distance and black-hole mass
simultaneously. Do not multiply marginalized posteriors with the same shared
prior twice. A joint generative model is the appropriate starting point.

The full SARM inference needs calibrated wavelength-dependent phases and
spectral information. Lab 07 supplies neither for its three RMIDs. A
successful ICCF is a useful ingredient, not a complete SARM distance.

### H4.4. Emissivity versus responsivity: the key astrophysical systematic

Spectroastrometry weights the brightness of gas in each spectral channel.
Reverberation weights the response of line emission to a changing continuum.
In schematic linear response, the local weight can be related to
$\partial j_\ell/\partial C$ rather than simply the emissivity $j_\ell$.
A radial-dependent responsivity can make the effective RM and SA radii differ.

For an idealized comparison, write $R_{\rm RM}=fR_{\rm SA}$ and suppose the
SA angular scale is $\theta_{\rm SA}=R_{\rm SA}/D_A$. A naive matched-radius
estimator would give

$$\widehat D_A=\frac{R_{\rm RM}}{\theta_{\rm SA}}=f D_A.$$

Thus a mismatch in physical scales directly biases the distance in this
idealized example. It is not enough for both observations to refer to the
same quasar. H$\beta$ and Pa$\alpha$ can trace different radial distributions,
and observations at different epochs can introduce additional differences.

Li et al. (2022) used velocity-resolved RM and SA for different lines with
shared central mass and inclination, allowing different BLR structures.
Li et al. (2025) explicitly incorporated radial responsivity in the joint
model. Other limitations include inclination/kinematics degeneracy,
continuum placement, finite spectral resolution, interferometric phase
calibration, temporal variability, and target selection for sufficient
brightness and angular size. A more elaborate model is not automatically
correct; posterior predictive checks and sensitivity to priors/geometry matter.

### H4.5. From SARM to cosmological parameters

For the same flat matter–$\Lambda$ background used in Labs 03–06,

$$D_A(z;H_0,\Omega_m)=\frac{c}{H_0(1+z)}\int_0^z
\frac{dz'}{\sqrt{\Omega_m(1+z')^3+1-\Omega_m}}.$$

For standard metric light propagation and photon-number conservation,
Etherington's reciprocity relation gives

$$D_L=(1+z)^2D_A.$$

This permits comparison to luminosity distances, but is not needed to fit
SARM angular distances directly. If $\Omega_m$ is fixed, the distance model
is proportional to $H_0^{-1}$. For low redshift only, $D_A\simeq cz/H_0$
is the leading-order approximation; at quasar redshifts use the full integral.
Peculiar velocities matter for nearby sources and should be included in a
precision analysis. A single distance does not independently determine both
$H_0$ and $\Omega_m$.

For a population, a hierarchical analysis should combine the individual
SARM likelihoods with a cosmological distance model and account for correlated
calibration, selection and modelling systematics. If using posterior samples
from published SARM fits, account for their original distance priors before
interpreting them as likelihoods. A table of marginal quantiles is not the
full likelihood.

**First-order uncertainty propagation** for matched positive scales:

$$\operatorname{Var}(\ln D_A)\approx\operatorname{Var}(\ln R)
+\operatorname{Var}(\ln\theta)-2\operatorname{Cov}(\ln R,\ln\theta).$$

For fixed cosmological shape and redshift, $\sigma_{\ln H_0}\approx
\sigma_{\ln D_A}$. Do not assume that all systematics average as $N^{-1/2}$.

### H4.6. Published results: evidence and uncertainty

| Study | Object(s) | Published result |
|---|---|---|
| Wang et al. (2020) | 3C 273 | $D_A=551.5^{+97.3}_{-78.7}$ Mpc; $H_0=71.5^{+11.9}_{-10.6}$ km s$^{-1}$ Mpc$^{-1}$ |
| GRAVITY Collaboration (2021) | NGC 3783 | $D_A=39.9^{+14.5}_{-11.9}$ Mpc |
| Li et al. (2022) | 3C 273, velocity-resolved SARM | $\log_{10}(D_A/\mathrm{Mpc})=2.83^{+0.32}_{-0.28}$ for their preferred model |
| Li et al. (2025) | Mrk 509, PDS 456, 3C 273, NGC 3783 | $H_0=69^{+12}_{-10}$ km s$^{-1}$ Mpc$^{-1}$ |

These are quoted study summaries, not independent data points to combine
naively. The 3C 273 analyses partly reuse observations; the four-object result
also revisits two previously studied targets. They demonstrate geometric
distance inference but do not resolve the Hubble tension. The original
forecast for a larger sample is a forecast, not a precision already achieved.

**Real published-summary exercise:** compare the reported 2020 and 2025
$H_0$ intervals below. The plot is a literature comparison, not a re-fit,
meta-analysis or reconstruction of the posterior distributions.

In [ ]:
# Published central values and asymmetric reported uncertainty offsets.
import numpy as np
import matplotlib.pyplot as plt

sarm_labels = ['Wang et al. (2020): 3C 273',
               'Li et al. (2025): four AGNs']
sarm_h0 = np.array([71.5, 69.0])
sarm_err = np.array([[10.6, 10.0], [11.9, 12.0]])
fig, ax = plt.subplots(figsize=(8, 2.8))
ax.errorbar(sarm_h0, np.arange(2), xerr=sarm_err,
            fmt='o', capsize=4)
ax.set_yticks(np.arange(2), labels=sarm_labels)
ax.invert_yaxis()
ax.set_xlabel(r'$H_0$ [km s$^{-1}$ Mpc$^{-1}$]')
ax.set_title('SARM: published Hubble-constant summaries')
ax.grid(axis='x', alpha=0.25)
fig.tight_layout()
plt.show()

for label, center, err in zip(sarm_labels, sarm_h0, sarm_err.T):
    lower, upper = center - err[0], center + err[1]
    print(f'{label}: [{lower:.1f}, {upper:.1f}]')

### H4.7. Optional numerical exercise using a published distance

Use the quoted 3C 273 distance from Wang et al. (2020). Hold the redshift and
$\Omega_m$ fixed, and transform the reported lower, central and upper
angular-distance values to $H_0$ using the flat cosmological distance formula.
For this classroom calculation use $z=0.158$ (rounded 3C 273 redshift) and
$\Omega_m=0.30$. This is an illustrative conditional transformation, not a
reproduction of their full SARM or cosmological fit. The transformation is
monotone decreasing, so the endpoint ordering reverses. If the original
summary values are quantiles, the transformed quantiles reverse accordingly.

Compare the exact transformation with the first-order relative uncertainty
formula. Explain why the transformed interval can be asymmetric.

In [ ]:
from scipy.integrate import quad

sarm_z = 0.158  # rounded redshift for a teaching calculation
sarm_omega = 0.30  # fixed teaching convention, not measured here
sarm_da = np.array([551.5-78.7, 551.5, 551.5+97.3])

def h0_from_da(da_mpc, z, omega_m=0.30):
    if np.any(np.asarray(da_mpc) <= 0):
        raise ValueError('Distance must be positive')
    integral = quad(lambda zp: 1/np.sqrt(
        omega_m*(1+zp)**3+1-omega_m), 0, z)[0]
    return 299792.458*integral/((1+z)*np.asarray(da_mpc))

sarm_h0_transformed = h0_from_da(sarm_da, sarm_z, sarm_omega)[::-1]
print('Input D_A lower, central, upper [Mpc]:', sarm_da)
print('Conditional H0 lower, central, upper:',
      np.round(sarm_h0_transformed, 2))
print('Not a new SARM fit; fixed redshift and cosmological shape.')
assert np.all(np.diff(sarm_h0_transformed)>0)

### H4.8. Questions for discussion and independent study

1. Derive the small-angle SARM distance and explain why the angle is in radians
   and the lag must be in the source frame.
2. Explain the difference between an integrated lag, a velocity-delay map,
   and a differential interferometric phase. Which quantities can Lab 07
   actually provide?
3. Why do the RM responsivity-weighted and SA emissivity-weighted radii need
   not be equal? Use the ratio $f$ above to explain the direction of a naive
   distance bias if $f>1$.
4. Which parameters are shared when fitting distinct emission lines? Why
   should one not impose identical radial structures without justification?
5. Compare the published Hubble-constant summaries. Explain why interval
   overlap is not a formal test of equality and why the two measurements are
   not independent.
6. Contrast SARM, RM R–L and UV–X-ray cosmology in terms of observables,
   distances, calibration, selection and dominant systematic uncertainties.

**Optional research extension:** read Li et al. (2022) and Li et al. (2025).
Describe a posterior predictive diagnostic for the velocity-resolved RM data
and one for interferometric phases, and propose a sensitivity test for
radial responsivity. No new fit or extra graded task is required.

# I. External literature comparison — only now consult published results
Open [Grier et al. 2017, Table 3](https://doi.org/10.3847/1538-4357/aa98dc).
Record the **observed-frame Hβ ICCF** lag and asymmetric uncertainty for each
assigned RMID, if available, and the paper's quality rating. Do not confuse
JAVELIN/CREAM or the adopted rest-frame lag in the erratum's Table 4 with the
observed-frame ICCF lag. Cite the table and method exactly. Convert to rest
frame using the same redshift if comparing rest-frame estimates. The release
has an erratum for derived quantities; consult the catalogue notes.

Compare your estimates with the reference, accounting for band/telescope
choice, intercalibration, sampling, search range, resampling and estimator
implementation. Agreement is a useful check, not independent validation on
new data, since the same observations are used. Disagreement need not imply
an error. Never tune the window until it matches the literature.

In [ ]:
# Student workspace: enter published Table 3 ICCF lags and errors (days).
# Leave NaN when not available. No hard-coded answers are supplied.
reference=pd.DataFrame({'RMID':IDS,'ref_obs':np.nan,
                        'ref_minus':np.nan,'ref_plus':np.nan,
                        'quality':np.nan})
comparison=rest_table.merge(reference,on='RMID',validate='one_to_one')
comparison['difference_obs']=comparison.obs50-comparison.ref_obs
comparison['ref_rest']=comparison.ref_obs/(1+comparison.z)
display(comparison.round(4))

# J. Graded extension — E1–E5
Complete every task; the supplied functions do not replace your reasoning.

**E1 — Data and estimator (all three objects).** Complete the five-line
statement, audit table, gap plots, prediction and ICCF interpretation. Report
both peak and centroid, rmax, overlap and edge flags; describe the sign and
prespecified search range. Explain two limitations of the interpolation.

**E2 — Uncertainty and robustness.** Submit the FR/RSS table, lag histograms,
second-seed comparison, alternative-window ICCFs and deletion diagnostics.
Include accepted/total, nonpositive fractions, observer- and rest-frame
quantiles and at least two quantitative comparisons. Explain why FR/RSS does
not test a red-noise null. Diagnose your most ambiguous object with the extra
test in F1. If fewer than 100 accepted draws remain, flag quantiles as unstable;
never replace a failure with a preferred lag.

**E3 — Literature benchmark.** Complete the reference table. Compare the same
line, method and frame, and explain discrepancies. Distinguish a shared-data
benchmark from validation on independent observations.

**E4 — Cosmological usability.** Create a decision table for each RMID:
`usable for further study / ambiguous / not usable`, evidence, main limitation,
and additional validation needed. Plot a forest-style comparison of observer-
frame medians and 68% FR/RSS intervals, with zero as reference and published
values in a separate marker style. Use a different marker or annotation for
ambiguous estimates and indicate missing intervals without inventing them.
Then propagate the lag-only distance-modulus uncertainty for usable objects.
Discuss R–L calibration, intrinsic scatter, selection, and the distinction from
UV–X-ray cosmology. These are candidate-quality decisions, not secure detections.

**Compulsory SARM concept (100–150 words).** Explain how combining reverberation
mapping with spectroastrometry can provide a geometric distance to a quasar.
Why is a reliable reverberation lag alone insufficient? Identify the distance
type and contrast the calibration requirements of SARM and the R–L method.
Explain one reason why the physical and angular scales must be modelled consistently.
Use the opening H4 discussion; no interferometric data or numerical SARM fit is required.

**E5 — Scientific conclusion (300–500 words).** State the estimand, observations,
model, numerical findings, uncertainty and robustness. Identify which lags (if
any) you would retain and why. Identify a limitation dominating each doubtful
result, compare with the publication, and explain what would be needed to
construct a defensible quasar Hubble diagram.

Required final sentence: “From this lab we can conclude ___, conditional on ___,
but we cannot conclude ___.” Full credit is possible with no accepted physical
lags if the reasoning is supported by the diagnostics.

In [ ]:
# Student workspace for E1–E4: add analysis cells, tables and figures.

### E5. Scientific conclusion

[Write your own 300–500-word conclusion here.]

# Submission and grading
Restart kernel and run all after completing the compulsory work. Optional H4
code may be run but its output is not required for assessment. Include the
ICCF validation-test output and your E4 conceptual SARM response. Include real data provenance, exact parameters,
seed, counts, figures with units and readable labels, and your own answers.
Submit `Surname_Name_L07.ipynb` plus the matching HTML/PDF export. Do not include
large raw catalogue files in your GitHub submission; keep the cache locally.

| Dimension | Weight | Evidence |
|---|---:|---|
| Scientific formulation | 20% | Defined estimand/frame, physical model, data and limits |
| Statistical correctness | 25% | Valid data cleaning, sign, overlap, ICCF and FR/RSS use |
| Uncertainty quantification | 20% | Quantiles, failed trials, seed, multimodality, propagation |
| Diagnostics / failure analysis | 15% | Sampling, robustness, difficult object, literature comparison |
| Reproducibility | 10% | Executed cells, versions, seeds, provenance, labels/export |
| Scientific interpretation | 10% | Quantitative conclusion, justified cosmological usability and SARM concept |

The same rubric as earlier graded labs applies. Do not reward agreement with a
published value alone. Distinguish supplied algorithm code from student work.

# References and reading pathway
## Required for the practical (selected sections, not entire papers)
- Grier, C. J., et al. (2017), ApJ **851**, 21, *The Sloan Digital Sky Survey
  Reverberation Mapping Project: Hα and Hβ Reverberation Measurements from
  First-year Spectroscopy and Photometry*.
  [DOI](https://doi.org/10.3847/1538-4357/aa98dc).
  Read the data/method sections first; Table 3 only after your analysis.
- [CDS ReadMe and data](https://cdsarc.cds.unistra.fr/viz-bin/cat/J/ApJ/851/21),
  [catalogue DOI](https://doi.org/10.26093/cds/vizier.18510021).
  Check units, flags and time conventions.
- Peterson, B. M., et al. (1998), PASP **110**, 660–670, *On Uncertainties in
  Cross-Correlation Lags and the Reality of Wavelength-dependent Continuum Lags
  in Active Galactic Nuclei*. [DOI](https://doi.org/10.1086/316177).
  Focus on flux randomization and random subset selection.
- Welsh, W. F. (1999), PASP **111**, 1347–1366, *On the Reliability of Cross
  Correlation Function Lag Determinations in Active Galactic Nuclei*.
  [Open preprint](https://arxiv.org/abs/astro-ph/9911112).
  Read the abstract and discussion of finite-baseline effects.

## Quasars as cosmological probes (required: Bentz, Watson abstracts and one critique)
- Bentz, M. C., et al. (2013), ApJ **767**, 149, *The Low-Luminosity End of the
  Radius–Luminosity Relationship for Active Galactic Nuclei*.
  [DOI](https://doi.org/10.1088/0004-637X/767/2/149).
  Hβ calibration and host-galaxy correction.
- Watson, D., Denney, K. D., Vestergaard, M., & Davis, T. M. (2011), ApJL
  **740**, L49, *A New Cosmological Distance Measure Using Active Galactic Nuclei*.
  [Open preprint](https://arxiv.org/abs/1109.4632).
- Czerny, B., et al. (2013), A&A **556**, A97, *Towards equation of state of dark
  energy from quasar monitoring: Reverberation strategy*.
  [DOI](https://doi.org/10.1051/0004-6361/201220832).
- *Standardizing reverberation-measured Mg II time-lag quasars, by using the
  radius–luminosity relation, and constraining cosmological model parameters*
  (2021), MNRAS **508**, 4722–4737.
  [DOI](https://doi.org/10.1093/mnras/stab2807). Joint calibration/cosmology fitting.
- *Do reverberation-measured Hβ quasars provide a useful test of cosmology?*
  (2022), MNRAS **513**, 1985–2005.
  [DOI](https://doi.org/10.1093/mnras/stac914). Read the abstract: limitations
  of Hβ constraints and accretion-rate corrections.
- Risaliti, G., & Lusso, E. (2019), Nature Astronomy **3**, 272–277,
  *Cosmological constraints from the Hubble diagram of quasars at high redshifts*.
  [Open preprint](https://arxiv.org/abs/1811.02590). Distinct UV–X-ray method.
- *Quasar UV/X-ray relation luminosity distances are shorter than
  reverberation-measured radius–luminosity relation luminosity distances*
  (2023), MNRAS **522**, 1247–1266 (see journal record for bibliographic details).
  [Article](https://academic.oup.com/mnras/article/522/1/1247/7111338).
  Comparison of the two probes; current systematic uncertainties matter.

## Optional stochastic-process material
- Kelly, B. C., Bechtold, J., & Siemiginowska, A. (2009), ApJ **698**, 895.
  [DOI](https://doi.org/10.1088/0004-637X/698/1/895). Continuous-time variability.
- Zu, Y., Kochanek, C. S., & Peterson, B. M. (2011), ApJ **735**, 80,
  *An Alternative Approach to Measuring Reverberation Lags in Active Galactic Nuclei*.
  [Open preprint](https://arxiv.org/abs/1008.0641). Joint stochastic response model.
- [Crete Gaussian Processes learning notebook](https://github.com/lionandjelka/2025_summer_school/blob/main/08_Gaussian_Processes/Gaussian_Processes.ipynb).



## SARM: advanced geometric quasar cosmology
- GRAVITY Collaboration (2018), Nature **563**, 657–660,
  *Spatially resolved rotation of the broad-line region of a quasar at sub-parsec scale*.
  [DOI](https://doi.org/10.1038/s41586-018-0731-9).
- Wang, J.-M., Songsheng, Y.-Y., Li, Y.-R., Du, P., & Zhang, Z.-X. (2020),
  Nature Astronomy **4**, 517–525, *A parallax distance to 3C 273 through
  spectroastrometry and reverberation mapping*.
  [DOI](https://doi.org/10.1038/s41550-019-0979-5),
  [preprint](https://arxiv.org/abs/1906.08417).
- GRAVITY Collaboration (2021), A&A **654**, A85,
  *A geometric distance to the supermassive black hole of NGC 3783*.
  [DOI](https://doi.org/10.1051/0004-6361/202141426).
- Li, Y.-R., et al. (2022), ApJ **927**, 58,
  *Spectroastrometry and Reverberation Mapping: the Mass and Geometric
  Distance of the Supermassive Black Hole in the Quasar 3C 273*.
  [Preprint](https://arxiv.org/abs/2201.04470).
- Li, Y.-R., et al. (2024), *SARM of Active Galactic Nuclei. I. The Hβ
  Broad-line Region Structure and Black Hole Mass of Five Quasars*.
  [Preprint](https://arxiv.org/abs/2407.08120).
- Li, Y.-R., Shangguan, J., Wang, J.-M., et al. (SARM Collaboration, 2025),
  ApJ **988**, 42, *Spectroastrometry and Reverberation Mapping of Active
  Galactic Nuclei. II. Measuring Geometric Distances and Black Hole Masses
  of Four Nearby Quasars*. [Preprint](https://arxiv.org/abs/2502.18856).
  Focus on radial responsivity and the combined cosmological constraint.

**Final synthesis:** a lag estimate is useful only in conjunction with sampling,
uncertainty, robustness, physical interpretation and population selection.
SARM additionally requires consistent angular and physical BLR models.
